Notebook to practice to construct a nice TPM data processing pipeline.

In [1]:
import pandas as pd
import numpy as np
import os
import glob
import sys

In [2]:
fcnts_path = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/pipeline_practice/Fcnts"

# Store folder name
files = os.listdir(fcnts_path)

# Filter out the summary files and keep only NDC comparisons
files = [csv for csv in files if ".summary" not in csv and "NDC0hr" in csv]

# Attach the folder to each of the files
files = ["".join([fcnts_path, "/" , csv]) for csv in files]

# Load each file as a dataframe
dfs = [pd.read_table(csv, sep = "\t", header = 0, skiprows = 1) for csv in files]

# Rename the columns of each pandas dataframe




In [3]:
practice_df = dfs[0]
practice_df.head(n = 5)



# Subset to the columns that either start with a "/" or are named "Genedid"





# Remember, we only need to get the NDC0hr columns from the 1st thing??
# Lowkey just process everything the same, then remove duplicate columns at the end to get rid of all the NDC?





# Set Gene names to the row names
practice_df = practice_df.set_index("Geneid")

# Select length column and those that correspond to the Fcnts values
practice_df = practice_df.loc[:, [col for col in list(practice_df.columns) if col == "Length" or col.startswith("/")]]

# Convert every entry of the dataframe to an integer to allow for integer manipulation
practice_df = practice_df.apply(lambda column: [int(entry) for entry in column])

# Divide Length by 1000 to get kb
practice_df["Length"] = practice_df["Length"].apply(lambda column: column / 1000)

# Select only columns with Fcnts
fcnts_cols = [col for col in list(practice_df.columns) if col != "Length"]

# Fcnts / gene length = (Counts per kb)
practice_df[fcnts_cols] = practice_df[fcnts_cols].apply(lambda column: column / practice_df["Length"])

# (Counts per kb) * 10^6 / (Total counts/kb) = TPM
practice_df[fcnts_cols] = practice_df[fcnts_cols].apply(lambda column: column * 10**6 / sum(column))

# Remove length column
practice_df.drop(columns = "Length", inplace = True)

# Rename the columns to remove extra crap

# can use df.rename(lambda x: ) to rename the samples?

# Obtain index of the last instance of / in the name of the thing
# Remove ".bam"
def sample_name_strip(name):
    """
    
    """
    # Find index of the last / 
    samplename_start_idx = name.strip().rfind("/") + 1
    
    # Subset 
    new_name = name[samplename_start_idx:]

    # Find index of . (.bam is at end of sample name) and remove filetag
    filetag_start_idx = new_name.rfind(".")
    new_name = new_name[:filetag_start_idx]
    
    # Remove "T4-wt"
    new_name = new_name.replace("T4-wt", "")

    return new_name

# Apply stripper to each column names
practice_df = practice_df.rename(columns = lambda column: sample_name_strip(column))

In [4]:

practice_df.T

Geneid,SP_0001,SP_0002,SP_0003,SP_0004,SP_0005,SP_0006,SP_0007,SP_0008,SP_0009,SP_0010,...,SP_2230,SP_2231,SP_2233,SP_2234,SP_2235,SP_2236,SP_2237,SP_2238,SP_2239,SP_2240
12CEF12CIP1hr-a,114.547926,73.295354,50.592072,161.505461,34.001150,27.607771,68.957388,86.661467,47.269891,64.144013,...,298.531351,78.570224,0.000000,24.984270,24.451026,24.846994,7.690736,24.225819,2450.455951,2051.693877
12CEF12CIP1hr-b,109.150760,102.588865,88.178438,149.196178,39.322556,28.670553,41.116912,115.286138,18.594538,71.371377,...,324.679202,81.702348,5.956063,32.011371,20.046542,18.283227,0.000000,41.930684,2475.206794,2085.234170
12CEF12CIP1hr-c,113.862974,84.181428,72.986338,190.529657,37.728815,28.045676,61.259319,117.381415,24.625472,71.844668,...,326.221497,74.388602,5.915885,15.060993,17.698945,19.416237,14.423491,33.444469,2671.713668,1988.560367
NDC0hr-a,102.614861,83.732058,54.760090,170.615323,26.057362,26.221659,60.187756,87.100980,17.816110,60.056247,...,140.721286,49.498498,14.583847,74.436121,17.784563,23.136758,1.932435,14.203398,1123.912135,1292.501235
NDC0hr-b,108.551217,105.322135,46.243152,182.004097,35.553683,33.067411,51.750764,84.876862,14.978270,76.582159,...,111.243416,51.055483,10.794886,65.312704,19.573197,23.966926,0.000000,28.786362,1092.615364,1319.242197
NDC0hr-c,95.498504,93.891570,42.709114,179.954974,30.138456,27.967309,39.829848,84.242992,16.626906,71.876976,...,134.324505,46.141131,14.912257,104.703579,21.184401,31.771686,6.492411,21.303224,1137.094714,1358.556260


In [10]:
cfu_path = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/pipeline_practice/all_cfus"

# Store folder name
cfu_files = os.listdir(cfu_path)

# Attach the folder to each of the files
cfu_files = ["".join([cfu_path, "/" , csv]) for csv in cfu_files]

# Load each file as a dataframe
cfu_dfs = [pd.read_table(csv, sep = ",", header = 0) for csv in cfu_files]

# Rename the columns of each pandas dataframe

cfu = cfu_dfs[0]

In [13]:
cfu.pivot(columns = "NDC1hr")

Triplicates                          14CEF1hr                \
NDC1hr  580000000  800000000  1000000000   580000000    800000000    
0                a        NaN        NaN  340000000.0          NaN   
1              NaN        NaN          b          NaN          NaN   
2              NaN          c        NaN          NaN  400000000.0   

                        13CEF1hr                               12CEF1hr  ...  \
NDC1hr   1000000000   580000000    800000000    1000000000   580000000   ...   
0               NaN  320000000.0          NaN          NaN  120000000.0  ...   
1       700000000.0          NaN          NaN  760000000.0          NaN  ...   
2               NaN          NaN  320000000.0          NaN          NaN  ...   

       34CEF13CIP1hr 34CEF12CIP1hr                         34CEF34CIP1hr  \
NDC1hr    1000000000    580000000   800000000   1000000000    580000000    
0                NaN    44000000.0         NaN         NaN    24000000.0   
1         33000000.0           NaN         NaN  14000000.0           NaN   
2                NaN           NaN  40000000.0         NaN           NaN   

                                    NDC0hr                           
NDC1hr  800000000   1000000000  580000000    800000000   1000000000  
0              NaN         NaN  38000000.0          NaN         NaN  
1              NaN  14000000.0         NaN          NaN  80000000.0  
2       21000000.0         NaN         NaN  100000000.0         NaN  

[3 rows x 84 columns]